# fMRIPrep → BrainVoyager conversion

Converts each subject/run into:
- `*.vtc` from preprocessed BOLD (NIfTI in MNI152NLin2009cAsym)
- `*_confounds.sdm` from `desc-confounds_timeseries.tsv`
- `*.prt` from behavioural CSV (`Subject_<id>_Run_<r>_RT.csv`)

Run the cells top-to-bottom. Configuration lives in **Cell 2** — edit paths/subjects/runs there.

## 1. Install / import dependencies

In [1]:
from pathlib import Path
import bvbabel
import nibabel as nib
import numpy as np
import pandas as pd

## 2. Configuration

Edit the paths, subjects, runs, TR, and task label below.

In [2]:
# --- Paths -------------------------------------------------------------------
FMRIPREP_DIR = Path("/Volumes/drive/AVP-BDD/derivatives/")
BEHAV_DIR    = Path("/Volumes/drive/AVP-BDD/behavior")
BV_DIR       = Path("/Volumes/drive/AVP-BDD/derivatives/brainvoyager")  # output

# --- Subjects & runs ---------------------------------------------------------
SUBJECTS = [
    # Group 1
    102, 103, 104, 105, 106, 107, 108, 109, 110, 111,
    112, 113, 115, 116, 117, 118, 119, 120, 121, 122,
    123, 124, 125, 126, 127, 128, 129, 130, 131, 132,
    # Group 2
    201, 202, 204, 205, 206, 207, 208, 209, 210, 211,
    212, 213, 214, 215, 216, 217, 218, 219, 220, 221,
    222, 223, 224, 225, 226, 227, 228, 229, 230,
]
RUNS = [1, 2, 3]

# --- Acquisition -------------------------------------------------------------
TR_SECONDS = 1.0                          # check your BOLD JSON sidecar!
TASK_LABEL = "SFlow"                      # BIDS task-<label>
SPACE      = "MNI152NLin2009cAsym"

# --- Confounds to include in SDM --------------------------------------------
MOTION_COLS         = ["trans_x", "trans_y", "trans_z",
                       "rot_x",   "rot_y",   "rot_z"]
EXTRA_COLS          = ["framewise_displacement"]
ACOMPCOR_N          = 6                   # top-N aCompCor components
COSINE_PREFIX       = "cosine"            # all cosine drift regressors
ADD_MOTION_OUTLIERS = True                # spike regressors

# --- Condition relabeling (CSV label -> BV PRT condition name) --------------
CONDITION_MAP = {
    "Condition 1": "lowSF-highC",
    "Condition 2": "highSF-highC",
    "Condition 3": "lowSF-lowC",
    "Condition 4": "highSF-lowC",
}
CONDITION_COLORS = {
    "lowSF-highC":  (255, 100, 100),
    "highSF-highC": (100, 255, 100),
    "lowSF-lowC":   (100, 100, 255),
    "highSF-lowC":  (255, 200, 100),
}

BV_DIR.mkdir(parents=True, exist_ok=True)
print(f"Will process {len(SUBJECTS)} subjects × {len(RUNS)} runs = "
      f"{len(SUBJECTS) * len(RUNS)} run(s) total")

Will process 59 subjects × 3 runs = 177 run(s) total


## 3. Conversion functions

In [3]:
# VMR cube origin: where MNI (0,0,0) sits in the BV 256³ template.
# Determined empirically from MNI_ICBM152_T1_NLIN_ASYM_09c_BRAIN.vmr.
# --- Tweakable VMR cube origin -----------------------------------------------
VMR_ORIGIN_X = 150   # leave this — looks right
VMR_ORIGIN_Y = 117   # leave this — top of brain looked OK
VMR_ORIGIN_Z = 114   # was 114 — bumping +20 to shift box posteriorly


def nifti_to_vtc(nifti_path: Path, vtc_path: Path, tr_ms: int) -> None:
    """Convert 4D NIfTI (MNI space) to VTC, framed against the BV MNI VMR."""
    img    = nib.load(str(nifti_path))
    data   = img.get_fdata(dtype=np.float32)        # (X, Y, Z, T) RAS+
    if data.ndim != 4:
        raise ValueError(f"{nifti_path} is not 4D")

    nx, ny, nz, nt = data.shape
    voxsizes = img.header.get_zooms()[:3]
    vtc_res  = int(round(voxsizes[0]))
    if not all(abs(v - vtc_res) < 0.05 for v in voxsizes):
        raise ValueError(f"Non-isotropic voxels {voxsizes} not supported by BV VTC")
    if vtc_res not in (1, 2, 3):
        raise ValueError(f"VTC resolution must be 1, 2, or 3 mm, got {vtc_res}")

    # MNI extent of the data using outer voxel corners
    corners = np.array([[-0.5,    -0.5,    -0.5,    1],
                        [nx-0.5,  ny-0.5,  nz-0.5,  1]])
    mni     = (img.affine @ corners.T).T[:, :3]
    mni_min, mni_max = mni.min(axis=0), mni.max(axis=0)

    # Map MNI -> BV cube using this VMR's measured origin.
    # BV LIP+: X=R->L (matches +MNI_X), Y=S->I (matches -MNI_Z), Z=A->P (matches -MNI_Y)
    XStart = int(round(VMR_ORIGIN_X - mni_max[0]))
    YStart = int(round(VMR_ORIGIN_Y - mni_max[2]))
    ZStart = int(round(VMR_ORIGIN_Z - mni_max[1]))
    XEnd   = XStart + nx * vtc_res
    YEnd   = YStart + nz * vtc_res
    ZEnd   = ZStart + ny * vtc_res

    header = {
        "File version": 3,
        "Source FMR name": vtc_path.stem + ".fmr",
        "Protocol attached": 0,
        "Protocol name": "",
        "Current protocol index": 0,
        "Data type (1:short int, 2:float)": 2,
        "Nr time points": nt,
        "VTC resolution relative to VMR (1, 2, or 3)": vtc_res,
        "XStart": XStart, "XEnd": XEnd,
        "YStart": YStart, "YEnd": YEnd,
        "ZStart": ZStart, "ZEnd": ZEnd,
        "L-R convention (0:unknown, 1:radiological, 2:neurological)": 1,
        "Reference space (0:unknown, 1:native, 2:ACPC, 3:Tal, 4:MNI)": 4,
        "TR (ms)": float(tr_ms),
    }
    bvbabel.vtc.write_vtc(str(vtc_path), header, data)
    print(f"  [VTC]  {vtc_path.name}  {vtc_res}mm  "
          f"X=[{XStart}:{XEnd}] Y=[{YStart}:{YEnd}] Z=[{ZStart}:{ZEnd}]")

In [4]:
def confounds_to_sdm(tsv_path: Path, sdm_path: Path) -> None:
    """Build an SDM of nuisance regressors from fMRIPrep confounds TSV."""
    df = pd.read_csv(tsv_path, sep="\t")

    cols = list(MOTION_COLS) + list(EXTRA_COLS)
    cols += sorted(c for c in df.columns if c.startswith("a_comp_cor_"))[:ACOMPCOR_N]
    cols += sorted(c for c in df.columns if c.startswith(COSINE_PREFIX))
    if ADD_MOTION_OUTLIERS:
        cols += sorted(c for c in df.columns if c.startswith("motion_outlier"))
    cols = [c for c in cols if c in df.columns]

    sub = df[cols].copy()
    sub = sub.fillna(sub.mean(numeric_only=True)).fillna(0.0)

    n_pred, n_tp = len(cols), len(sub)

    header = {
        "FileVersion":            1,
        "NrOfPredictors":         n_pred,
        "NrOfDataPoints":         n_tp,
        "IncludesConstant":       0,
        "FirstConfoundPredictor": 1,   # 1-indexed: all predictors are confounds
    }

    # bvbabel expects a list of per-predictor dicts
    data_sdm = []
    for name in cols:
        data_sdm.append({
            "NameOfPredictor":    name,
            "ColorOfPredictor":   [120, 120, 120],
            "ValuesOfPredictor":  sub[name].to_numpy(dtype=np.float32).tolist(),
        })

    bvbabel.sdm.write_sdm(str(sdm_path), header, data_sdm)
    print(f"  [SDM]  {sdm_path.name}  {n_tp} TPs x {n_pred} regressors")

In [5]:
def behav_to_prt(csv_path: Path, prt_path: Path) -> None:
    """Convert behavioural CSV to BV PRT (block design, msec resolution)."""
    df = pd.read_csv(csv_path)

    blocks = (df.groupby("Block")
                .agg(onset=("Stimulus Onset (s)",  "min"),
                     offset=("Stimulus Offset (s)", "max"),
                     condition=("Condition",        "first"),
                     n_conds=("Condition", lambda x: x.nunique()))
                .reset_index())

    if (blocks["n_conds"] != 1).any():
        bad = blocks[blocks["n_conds"] != 1]["Block"].tolist()
        raise ValueError(f"Block(s) {bad} contain multiple conditions in {csv_path}")

    blocks["cond_label"] = blocks["condition"].map(CONDITION_MAP)
    if blocks["cond_label"].isna().any():
        unknown = blocks[blocks["cond_label"].isna()]["condition"].unique().tolist()
        raise ValueError(f"Unmapped condition(s) {unknown} in {csv_path}")

    blocks["onset_ms"]  = (blocks["onset"]  * 1000.0).round().astype(int)
    blocks["offset_ms"] = (blocks["offset"] * 1000.0).round().astype(int)

    cond_list = []
    for cond_name in CONDITION_MAP.values():
        sub = blocks[blocks["cond_label"] == cond_name].sort_values("onset_ms")
        if len(sub) == 0:
            continue
        intervals = sub[["onset_ms", "offset_ms"]].to_numpy(dtype=np.int32)
        cond_list.append({
            "NameOfCondition": cond_name,
            "NrOfOccurances":  len(sub),
            "Time start":      intervals[:, 0].tolist(),
            "Time stop":       intervals[:, 1].tolist(),
            "Color":           list(CONDITION_COLORS[cond_name]),
        })

    header = {
        "FileVersion":         2,
        "ResolutionOfTime":    "msec",
        "Experiment":          prt_path.stem,
        "BackgroundColor":     "0 0 0",
        "TextColor":           "255 255 255",
        "TimeCourseColor":     "255 255 255",
        "TimeCourseThick":     3,
        "ReferenceFuncColor":  "0 0 80",
        "ReferenceFuncThick":  3,
        "NrOfConditions":      len(cond_list),
    }
    bvbabel.prt.write_prt(str(prt_path), header, cond_list)
    print(f"  [PRT]  {prt_path.name}  {len(cond_list)} conditions, {len(blocks)} blocks")

## 4. Per-subject driver

In [5]:
def process_subject(sub_num, runs, fmriprep_dir, behav_dir, bv_dir):
    """Process one subject across the requested runs."""
    sub_id     = f"sub-{sub_num}"
    raw_sub_id = str(sub_num)

    sub_root = fmriprep_dir / sub_id
    sub_out  = bv_dir / sub_id
    sub_out.mkdir(parents=True, exist_ok=True)

    print(f"\n=== {sub_id} ===")
    if not sub_root.exists():
        print(f"  [SKIP all] missing subject dir: {sub_root}")
        return

    for run in runs:
        run_str = f"run-{run:02d}"  # zero-padded: run-01, run-02, ...

        # Glob handles optional ses-XX/ in path and filename
        bold_glob = f"**/{sub_id}*_task-{TASK_LABEL}_{run_str}_space-{SPACE}_desc-preproc_bold.nii.gz"
        conf_glob = f"**/{sub_id}*_task-{TASK_LABEL}_{run_str}_desc-confounds_timeseries.tsv"

        bold_hits = sorted(sub_root.glob(bold_glob))
        conf_hits = sorted(sub_root.glob(conf_glob))

        bold_nii  = bold_hits[0] if bold_hits else None
        conf_tsv  = conf_hits[0] if conf_hits else None
        behav_csv = behav_dir / raw_sub_id / "low-level" / f"Subject_{raw_sub_id}_Run_{run}_RT.csv"

        missing = []
        if not bold_nii:           missing.append(f"BOLD matching {bold_glob}")
        if not conf_tsv:           missing.append(f"confounds matching {conf_glob}")
        if not behav_csv.exists(): missing.append(f"behav {behav_csv}")
        if missing:
            print(f"  [SKIP run-{run}] missing: {', '.join(missing)}")
            continue

        if len(bold_hits) > 1:
            print(f"  [WARN run-{run}] multiple BOLDs matched, using {bold_nii.name}")

        stem = f"{sub_id}_task-{TASK_LABEL}_{run_str}"
        try:
            #nifti_to_vtc    (bold_nii,  sub_out / f"{stem}.vtc",
                           #  tr_ms=int(TR_SECONDS * 1000))
            confounds_to_sdm(conf_tsv,  sub_out / f"{stem}_confounds.sdm")
            behav_to_prt    (behav_csv, sub_out / f"{stem}.prt")
        except Exception as e:
            print(f"  [ERROR run-{run}] {type(e).__name__}: {e}")

## 5. (Optional) Single-subject smoke test

Strongly recommended before running the full batch — check VTC framing, design matrix, and beta map for one subject in BV first.

In [36]:
process_subject(102, RUNS, FMRIPREP_DIR, BEHAV_DIR, BV_DIR)


=== sub-102 ===
  [VTC]  sub-102_task-SFlow_run-01.vtc  3mm  X=[70:229] Y=[28:196] Z=[34:229]
  [SDM]  sub-102_task-SFlow_run-01_confounds.sdm  360 TPs x 87 regressors
  [PRT]  sub-102_task-SFlow_run-01.prt  4 conditions, 16 blocks
  [VTC]  sub-102_task-SFlow_run-02.vtc  3mm  X=[70:229] Y=[28:196] Z=[34:229]
  [SDM]  sub-102_task-SFlow_run-02_confounds.sdm  360 TPs x 52 regressors
  [PRT]  sub-102_task-SFlow_run-02.prt  4 conditions, 16 blocks
  [VTC]  sub-102_task-SFlow_run-03.vtc  3mm  X=[70:229] Y=[28:196] Z=[34:229]
  [SDM]  sub-102_task-SFlow_run-03_confounds.sdm  360 TPs x 26 regressors
  [PRT]  sub-102_task-SFlow_run-03.prt  4 conditions, 16 blocks


In [37]:
import nibabel as nib
import numpy as np

nii = nib.load("/Volumes/drive/AVP-BDD/derivatives/sub-102/ses-01/func/"
               "sub-102_ses-01_task-SFlow_run-01_space-MNI152NLin2009cAsym_desc-preproc_bold.nii.gz")

print("Shape (XYZT):", nii.shape)
print("Voxel sizes :", nii.header.get_zooms()[:3])
print("Axis codes  :", nib.orientations.aff2axcodes(nii.affine))
print("\nFor each NIfTI axis, the MNI extent it spans:")
for i, name in enumerate("XYZ"):
    n = nii.shape[i]
    vec = nii.affine[:3, i]            # mm/voxel direction in MNI
    extent = abs(n * vec).sum()
    direction = "RL" if vec[0] != 0 else "AP" if vec[1] != 0 else "IS"
    print(f"  axis {i} (n={n:3d}, voxsize {nii.header.get_zooms()[i]}): "
          f"extent {extent:.0f} mm, direction {direction}")

Shape (XYZT): (53, 65, 56, 360)
Voxel sizes : (3.0, 3.0, 3.0)
Axis codes  : ('R', 'A', 'S')

For each NIfTI axis, the MNI extent it spans:
  axis 0 (n= 53, voxsize 3.0): extent 159 mm, direction RL
  axis 1 (n= 65, voxsize 3.0): extent 195 mm, direction AP
  axis 2 (n= 56, voxsize 3.0): extent 168 mm, direction IS


In [38]:
import nibabel as nib
import numpy as np

nii = nib.load("/Volumes/drive/AVP-BDD/derivatives/sub-102/ses-01/func/"
               "sub-102_ses-01_task-SFlow_run-01_space-MNI152NLin2009cAsym_desc-preproc_bold.nii.gz")
d = nii.get_fdata()
print(f"Shape: {d.shape}    dtype: {d.dtype}")
print(f"Global stats: min={d.min():.2f} max={d.max():.2f} mean={d.mean():.2f}")

# Per-volume mean — flat or zero volumes show as dropouts in time
vol_means = d.mean(axis=(0,1,2))
print(f"\nPer-volume mean (T={len(vol_means)}):")
print(f"  range  {vol_means.min():.1f} .. {vol_means.max():.1f}")
print(f"  median {np.median(vol_means):.1f}")
zero_vols = np.where(vol_means < 1)[0]
print(f"  volumes with mean < 1: {len(zero_vols)}  {zero_vols[:20]}")

# Check for empty slices/columns within a sample volume
print(f"\nWithin volume 100, looking for missing planes:")
v = d[..., 100]
for axis, name in [(0, "X (sagittal)"), (1, "Y (coronal)"), (2, "Z (axial)")]:
    plane_means = v.mean(axis=tuple(i for i in range(3) if i != axis))
    empty = np.where(plane_means < 1)[0]
    print(f"  empty {name} planes: {len(empty)} / {v.shape[axis]}")

# Voxels that are zero across ALL time
all_zero = (d == 0).all(axis=3)
print(f"\nVoxels zero across all time: {all_zero.sum()} / {all_zero.size} "
      f"({100*all_zero.mean():.1f}%)")

# A spatial sample: pick a voxel solidly in the brain (motor cortex-ish)
ix, iy, iz = 26, 40, 40
tc = d[ix, iy, iz, :]
print(f"\nSample voxel ({ix},{iy},{iz}) time course:")
print(f"  range {tc.min():.1f} .. {tc.max():.1f}, mean {tc.mean():.1f}, "
      f"std {tc.std():.2f}")
print(f"  first 5 timepoints: {tc[:5]}")

Shape: (53, 65, 56, 360)    dtype: float64
Global stats: min=-1662.46 max=30028.00 mean=3600.17

Per-volume mean (T=360):
  range  3561.6 .. 3641.6
  median 3598.4
  volumes with mean < 1: 0  []

Within volume 100, looking for missing planes:
  empty X (sagittal) planes: 0 / 53
  empty Y (coronal) planes: 0 / 65
  empty Z (axial) planes: 0 / 56

Voxels zero across all time: 8781 / 192920 (4.6%)

Sample voxel (26,40,40) time course:
  range 7239.9 .. 9371.7, mean 7873.2, std 374.89
  first 5 timepoints: [8573.71875    9371.72363281 9165.16015625 9059.30078125 9032.56445312]


## 6. Run the full batch

In [40]:
from datetime import datetime
t0 = datetime.now()
for sub_num in SUBJECTS:
    process_subject(sub_num, RUNS, FMRIPREP_DIR, BEHAV_DIR, BV_DIR)
print(f"\nDone in {datetime.now() - t0}")


=== sub-102 ===
  [SDM]  sub-102_task-SFlow_run-01_confounds.sdm  360 TPs x 87 regressors
  [PRT]  sub-102_task-SFlow_run-01.prt  4 conditions, 16 blocks
  [SDM]  sub-102_task-SFlow_run-02_confounds.sdm  360 TPs x 52 regressors
  [PRT]  sub-102_task-SFlow_run-02.prt  4 conditions, 16 blocks
  [SDM]  sub-102_task-SFlow_run-03_confounds.sdm  360 TPs x 26 regressors
  [PRT]  sub-102_task-SFlow_run-03.prt  4 conditions, 16 blocks

=== sub-103 ===
  [SDM]  sub-103_task-SFlow_run-01_confounds.sdm  360 TPs x 62 regressors
  [PRT]  sub-103_task-SFlow_run-01.prt  4 conditions, 16 blocks
  [SDM]  sub-103_task-SFlow_run-02_confounds.sdm  360 TPs x 48 regressors
  [PRT]  sub-103_task-SFlow_run-02.prt  4 conditions, 16 blocks
  [SDM]  sub-103_task-SFlow_run-03_confounds.sdm  360 TPs x 44 regressors
  [PRT]  sub-103_task-SFlow_run-03.prt  4 conditions, 16 blocks

=== sub-104 ===
  [SDM]  sub-104_task-SFlow_run-01_confounds.sdm  360 TPs x 54 regressors
  [PRT]  sub-104_task-SFlow_run-01.prt  4 condi

## 7. Quick check: what got written?

In [ ]:
rows = []
for sub_num in SUBJECTS:
    sub_dir = BV_DIR / f"sub-{sub_num}"
    if not sub_dir.exists():
        rows.append({"sub": sub_num, "vtc": 0, "sdm": 0, "prt": 0}); continue
    rows.append({
        "sub": sub_num,
        "vtc": len(list(sub_dir.glob("*.vtc"))),
        "sdm": len(list(sub_dir.glob("*.sdm"))),
        "prt": len(list(sub_dir.glob("*.prt"))),
    })
summary = pd.DataFrame(rows)
expected = len(RUNS)
incomplete = summary[(summary.vtc != expected) | (summary.sdm != expected) | (summary.prt != expected)]
print(f"Expected {expected} of each per subject.")
print(f"Subjects with all files complete: {len(summary) - len(incomplete)}/{len(summary)}")
if len(incomplete):
    print("\nIncomplete subjects:")
    print(incomplete.to_string(index=False))

In [6]:
import bvbabel
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.special import gamma as gamma_fn


def double_gamma_hrf(t, a1=6.0, b1=1.0, a2=16.0, b2=1.0, c=1/6.0):
    """SPM-style canonical double-gamma HRF, sampled at times t (seconds)."""
    h = ((t**a1 * np.exp(-t/b1)) / (b1**(a1+1) * gamma_fn(a1+1))
         - c * (t**a2 * np.exp(-t/b2)) / (b2**(a2+1) * gamma_fn(a2+1)))
    h[t < 0] = 0
    return h


def build_combined_sdm(prt_path: Path, sdm_path: Path, out_path: Path,
                       n_timepoints: int, tr_seconds: float) -> None:
    """Combine PRT (HRF-convolved task) + confound SDM into a single SDM."""

    # --- Task predictors from PRT ------------------------------------------
    prt_header, prt_conds = bvbabel.prt.read_prt(str(prt_path))

    fine_dt = 0.05                    # 50 ms convolution grid
    duration = n_timepoints * tr_seconds
    fine_t   = np.arange(0, duration, fine_dt)
    hrf      = double_gamma_hrf(np.arange(0, 32, fine_dt))

    task_predictors = []
    for cond in prt_conds:
        stick = np.zeros_like(fine_t)
        starts = np.atleast_1d(cond["Time start"]).astype(float) / 1000.0
        stops  = np.atleast_1d(cond["Time stop"]).astype(float)  / 1000.0
        for s, e in zip(starts, stops):
            i0, i1 = int(s/fine_dt), int(e/fine_dt)
            stick[i0:i1] = 1.0
        conv = np.convolve(stick, hrf)[:len(fine_t)]
        # Downsample: take the value at each TR's center
        tr_indices = ((np.arange(n_timepoints) + 0.5) * tr_seconds / fine_dt).astype(int)
        tr_indices = np.clip(tr_indices, 0, len(conv) - 1)
        task_predictors.append({
            "NameOfPredictor":   cond["NameOfCondition"],
            "ColorOfPredictor":  list(cond["Color"]),
            "ValuesOfPredictor": conv[tr_indices].astype(np.float32).tolist(),
        })

    # --- Confound predictors from existing SDM -----------------------------
    sdm_header, sdm_predictors = bvbabel.sdm.read_sdm(str(sdm_path))

    # --- Combine ------------------------------------------------------------
    all_predictors = task_predictors + sdm_predictors
    n_pred = len(all_predictors)

    combined_header = {
        "FileVersion":            1,
        "NrOfPredictors":         n_pred,
        "NrOfDataPoints":         n_timepoints,
        "IncludesConstant":       0,
        "FirstConfoundPredictor": len(task_predictors) + 1,  # 1-indexed
    }
    bvbabel.sdm.write_sdm(str(out_path), combined_header, all_predictors)
    print(f"  [DESIGN]  {out_path.name}  {n_timepoints} TPs x {n_pred} predictors "
          f"({len(task_predictors)} task + {len(sdm_predictors)} confounds)")


# --- Build combined SDMs for every subject/run -------------------------------
N_TIMEPOINTS = 360                    # check your VTC; should match TR count
TR_SECONDS_  = TR_SECONDS              # use the value from the config cell

print(f"Building combined design matrices ({N_TIMEPOINTS} TPs, TR={TR_SECONDS_}s)\n")
for sub_num in SUBJECTS:
    sub_dir = BV_DIR / f"sub-{sub_num}"
    if not sub_dir.exists():
        continue
    print(f"=== sub-{sub_num} ===")
    for run in RUNS:
        rr  = f"run-{run:02d}"
        prt = sub_dir / f"sub-{sub_num}_task-{TASK_LABEL}_{rr}.prt"
        sdm = sub_dir / f"sub-{sub_num}_task-{TASK_LABEL}_{rr}_confounds.sdm"
        out = sub_dir / f"sub-{sub_num}_task-{TASK_LABEL}_{rr}_design.sdm"

        if not prt.exists() or not sdm.exists():
            print(f"  [SKIP {rr}] missing PRT or confounds SDM"); continue
        try:
            build_combined_sdm(prt, sdm, out, N_TIMEPOINTS, TR_SECONDS_)
        except Exception as e:
            print(f"  [ERROR {rr}] {type(e).__name__}: {e}")

Building combined design matrices (360 TPs, TR=1.0s)

=== sub-102 ===
  [DESIGN]  sub-102_task-SFlow_run-01_design.sdm  360 TPs x 91 predictors (4 task + 87 confounds)
  [DESIGN]  sub-102_task-SFlow_run-02_design.sdm  360 TPs x 56 predictors (4 task + 52 confounds)
  [DESIGN]  sub-102_task-SFlow_run-03_design.sdm  360 TPs x 30 predictors (4 task + 26 confounds)
=== sub-103 ===
  [DESIGN]  sub-103_task-SFlow_run-01_design.sdm  360 TPs x 66 predictors (4 task + 62 confounds)
  [DESIGN]  sub-103_task-SFlow_run-02_design.sdm  360 TPs x 52 predictors (4 task + 48 confounds)
  [DESIGN]  sub-103_task-SFlow_run-03_design.sdm  360 TPs x 48 predictors (4 task + 44 confounds)
=== sub-104 ===
  [DESIGN]  sub-104_task-SFlow_run-01_design.sdm  360 TPs x 58 predictors (4 task + 54 confounds)
  [DESIGN]  sub-104_task-SFlow_run-02_design.sdm  360 TPs x 62 predictors (4 task + 58 confounds)
  [DESIGN]  sub-104_task-SFlow_run-03_design.sdm  360 TPs x 106 predictors (4 task + 102 confounds)
=== sub-105 ==

In [14]:
### design matrix files wout motion regressors (just including spikes w excessive motion)

import bvbabel
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.special import gamma as gamma_fn


def double_gamma_hrf(t, a1=6.0, b1=1.0, a2=16.0, b2=1.0, c=1/6.0):
    """SPM-style canonical double-gamma HRF, sampled at times t (seconds)."""
    h = ((t**a1 * np.exp(-t/b1)) / (b1**(a1+1) * gamma_fn(a1+1))
         - c * (t**a2 * np.exp(-t/b2)) / (b2**(a2+1) * gamma_fn(a2+1)))
    h[t < 0] = 0
    return h


def is_motion_spike(predictor_name: str) -> bool:
    """Identify motion-outlier spike regressors in the fMRIPrep confounds SDM.
    fMRIPrep names them 'motion_outlier00', 'motion_outlier01', etc."""
    return predictor_name.startswith("motion_outlier")


def build_combined_sdm(prt_path: Path, sdm_path: Path, out_path: Path,
                       n_timepoints: int, tr_seconds: float) -> None:
    """Combine PRT (HRF-convolved task) + motion-spike-only confounds into one SDM."""

    # --- Task predictors from PRT ------------------------------------------
    prt_header, prt_conds = bvbabel.prt.read_prt(str(prt_path))

    fine_dt  = 0.05
    duration = n_timepoints * tr_seconds
    fine_t   = np.arange(0, duration, fine_dt)
    hrf      = double_gamma_hrf(np.arange(0, 32, fine_dt))

    task_predictors = []
    for cond in prt_conds:
        stick  = np.zeros_like(fine_t)
        starts = np.atleast_1d(cond["Time start"]).astype(float) / 1000.0
        stops  = np.atleast_1d(cond["Time stop"]).astype(float)  / 1000.0
        for s, e in zip(starts, stops):
            stick[int(s/fine_dt):int(e/fine_dt)] = 1.0
        conv = np.convolve(stick, hrf)[:len(fine_t)]
        tr_idx = ((np.arange(n_timepoints) + 0.5) * tr_seconds / fine_dt).astype(int)
        tr_idx = np.clip(tr_idx, 0, len(conv) - 1)
        task_predictors.append({
            "NameOfPredictor":   cond["NameOfCondition"],
            "ColorOfPredictor":  list(cond["Color"]),
            "ValuesOfPredictor": conv[tr_idx].astype(np.float32).tolist(),
        })

    # --- Confound predictors: KEEP ONLY motion-outlier spikes --------------
    _, all_confounds = bvbabel.sdm.read_sdm(str(sdm_path))
    spike_predictors = [p for p in all_confounds if is_motion_spike(p["NameOfPredictor"])]

    # --- Combine and write -------------------------------------------------
    all_predictors = task_predictors + spike_predictors
    n_pred = len(all_predictors)

    header = {
        "FileVersion":            1,
        "NrOfPredictors":         n_pred,
        "NrOfDataPoints":         n_timepoints,
        "IncludesConstant":       0,
        "FirstConfoundPredictor": len(task_predictors) + 1,
    }
    bvbabel.sdm.write_sdm(str(out_path), header, all_predictors)
    print(f"  [DESIGN]  {out_path.name}  "
          f"{len(task_predictors)} task + {len(spike_predictors)} spikes")


# --- Run the batch (writes *_design-spikesonly.sdm files) --------------------
N_TIMEPOINTS = 360

print(f"Building motion-spike-only design matrices\n")
for sub_num in SUBJECTS:
    sub_dir = BV_DIR / f"sub-{sub_num}"
    if not sub_dir.exists():
        continue
    print(f"=== sub-{sub_num} ===")
    for run in RUNS:
        rr  = f"run-{run:02d}"
        prt = sub_dir / f"sub-{sub_num}_task-{TASK_LABEL}_{rr}.prt"
        sdm = sub_dir / f"sub-{sub_num}_task-{TASK_LABEL}_{rr}_confounds.sdm"
        out = sub_dir / f"sub-{sub_num}_task-{TASK_LABEL}_{rr}_design-spikesonly.sdm"
        if not prt.exists() or not sdm.exists():
            print(f"  [SKIP {rr}] missing PRT or confounds SDM"); continue
        try:
            build_combined_sdm(prt, sdm, out, N_TIMEPOINTS, TR_SECONDS)
        except Exception as e:
            print(f"  [ERROR {rr}] {type(e).__name__}: {e}")

Building motion-spike-only design matrices

=== sub-102 ===
  [DESIGN]  sub-102_task-SFlow_run-01_design-spikesonly.sdm  4 task + 70 spikes
  [DESIGN]  sub-102_task-SFlow_run-02_design-spikesonly.sdm  4 task + 35 spikes
  [DESIGN]  sub-102_task-SFlow_run-03_design-spikesonly.sdm  4 task + 9 spikes
=== sub-103 ===
  [DESIGN]  sub-103_task-SFlow_run-01_design-spikesonly.sdm  4 task + 45 spikes
  [DESIGN]  sub-103_task-SFlow_run-02_design-spikesonly.sdm  4 task + 31 spikes
  [DESIGN]  sub-103_task-SFlow_run-03_design-spikesonly.sdm  4 task + 27 spikes
=== sub-104 ===
  [DESIGN]  sub-104_task-SFlow_run-01_design-spikesonly.sdm  4 task + 37 spikes
  [DESIGN]  sub-104_task-SFlow_run-02_design-spikesonly.sdm  4 task + 41 spikes
  [DESIGN]  sub-104_task-SFlow_run-03_design-spikesonly.sdm  4 task + 85 spikes
=== sub-105 ===
  [DESIGN]  sub-105_task-SFlow_run-01_design-spikesonly.sdm  4 task + 12 spikes
  [DESIGN]  sub-105_task-SFlow_run-02_design-spikesonly.sdm  4 task + 4 spikes
  [DESIGN]  su

In [9]:
BV_DIR

PosixPath('/Volumes/drive/AVP-BDD/derivatives/brainvoyager')

In [16]:
mdm_path = BV_DIR / "group_RFX-nomotion.mdm"

study_lines = []
for sub_num in SUBJECTS:
    sub_dir = BV_DIR / f"sub-{sub_num}"
    if not sub_dir.exists():
        continue
    for run in RUNS:
        rr  = f"run-{run:02d}"
        vtc = sub_dir / f"sub-{sub_num}_ses-01_task-{TASK_LABEL}_{rr}_space-MNI152NLin2009cAsym_desc-preproc_bold.vtc"
        sdm = sub_dir / f"sub-{sub_num}_task-{TASK_LABEL}_{rr}_design-spikesonly.sdm"
        if vtc.exists() and sdm.exists():
            study_lines.append(f'"{vtc}" "{sdm}"')
        else:
            print(f"WARN: missing files for sub-{sub_num} {rr}")

# Match BV's exact format (from the BV-native MDM you sent)
mdm_text = (
    "\n"                                           # leading blank line
    "FileVersion:          3\n"
    "TypeOfFunctionalData: VTC\n\n"
    "RFX-GLM:              1\n\n"
    "PSCTransformation:    0\n"
    "zTransformation:      1\n"
    "SeparatePredictors:   2\n\n"
    f"NrOfStudies:          {len(study_lines)}\n"  # NO blank line after this
    + "\n".join(study_lines) + "\n"
)
mdm_path.write_text(mdm_text)
print(f"Wrote {mdm_path} with {len(study_lines)} studies")

Wrote /Volumes/drive/AVP-BDD/derivatives/brainvoyager/group_RFX-nomotion.mdm with 177 studies


# Redoing for High-level 